# ДЗ-1. Ваш участок ПРАЙМ

Вы — аналитик команды подписки ПРАЙМ. У каждого аналитика свой **участок**: одна таблица, один сегмент и одно окно времени. Это тот самый персональный срез из `prime.assignments`, с которым вы работали на первом семинаре, и больше он ни у кого не повторяется.

В понедельник руководитель просит сводку по участку — восемь задач. В каждой две части: **расчёт** (функция на Python) и **вопрос**, на который вы отвечаете словами в ячейке «✍️ Ответ на вопрос N». Числа у всех разные, поэтому и выводы у каждого свои.

### Как устроен ноутбук

| Часть | Что там | Что делаете вы |
|---|---|---|
| **1. Подготовка** | готовый код: подключение к базе и выгрузка вашего участка | вписываете пять значений в ячейку 1.1 и запускаете ячейки по порядку |
| **2. Восемь задач** | условия, заготовки функций, вопросы | пишете код функций и ответы словами |
| **3. Итог** | готовый код: сводка ваших ответов и проверка их формата | запускаете перед сдачей |
| **4. Исследование** | две открытые задачи для тех, кто хочет 9 или 10 | по желанию |

**Оценка.** Части 1–3 — это оценка до 8 баллов: сделали всё аккуратно, выводы верные — 8, и это «отлично». 9 и 10 — только за исследование в части 4. Все критерии — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-1-окружение-и-репозиторий), там же — как сдавать.

Запускайте ячейки сверху вниз (`Shift + Enter`). Перед сдачей — **Kernel → Restart Kernel and Run All Cells**: ноутбук должен пройти целиком без красного.

Ноутбук лежит в **вашем** репозитории как `notebooks/hw1.ipynb`, а JupyterLab запускается из корня `prime-monitor` командой `uv run jupyter lab`.

## Часть 1. Подготовка — готовый код

Код в этой части писать не нужно — он готов. Единственная ваша правка — пять значений в ячейке 1.1. Номер ячейки — это номер заголовка над ней: «ячейка 1.1» — код сразу под заголовком «1.1 Ваш участок».

Код длинный, потому что аккуратно обходит несколько ловушек, до которых мы дойдём позже. Прочитать его полезно, разбирать каждую строку не обязательно.

| Ячейка | Что делает | Что делаете вы |
|---|---|---|
| 1.1 Ваш участок | хранит пять значений участка | вписываете значения |
| 1.2 Подключение | подключается к базе через ваш `.env` | запускаете |
| 1.3 Строка участка | показывает вашу строку из `prime.assignments` | запускаете, копируете значения в 1.1 и запускаете 1.1 ещё раз |
| 1.4 Запрос | описывает, как достать участок из любой из четырёх таблиц | запускаете |
| 1.5 Выгрузка | достаёт участок в переменную `rows` | запускаете — дальше вы работаете с `rows` |
| 1.6 Формат ответов | служебная функция для итога | запускаете |

### 1.1 Ваш участок

Впишите значения из своей строки `prime.assignments` — строками, в кавычках. Какие они, покажет ячейка 1.3: запустите 1.2 и 1.3, скопируйте напечатанные строки сюда и **запустите эту ячейку ещё раз**, иначе Python продолжит видеть старые значения.

Это **единственное место**, где указан участок. При проверке преподаватель подставит сюда другой участок — и все ответы должны пересчитаться сами. Поэтому ячейку не удаляйте и не пересоздавайте, а значения больше нигде не повторяйте.

In [1]:
TABLE = 'service_usage'          # таблица, например "payments"
SEGMENT_COLUMN = 'device_type'  # колонка сегмента, например "payment_method"
SEGMENT = 'desktop'         # сегмент — значение в этой колонке, например "card"
PERIOD_START = '2025-04-01'    # первый день окна, например "2026-04-01"
PERIOD_END = '2025-04-30'      # последний день окна ВКЛЮЧИТЕЛЬНО, например "2026-06-30"

### 1.2 Подключение

Подключаемся к учебной базе и заводим функцию `query(sql)`: она отправляет SQL-запрос и возвращает результат списком словарей — одна строка результата, один словарь, как на семинарах. Пароля в ноутбуке нет: его знает только ваш `.env`.

Если здесь ошибка `Пакет prime не найден` — ноутбук запущен не в окружении вашего репозитория. Если `Не найдена переменная PRIME_DSN` — нет файла `.env` в корне `prime-monitor`.

Соединение открывается на каждый запрос и сразу закрывается: так после сна ноутбука или смены Wi-Fi ничего не отваливается, а общий сервер не держит лишних подключений.

In [2]:
import sys
from collections import Counter, defaultdict  # понадобятся в задачах
from datetime import date
from decimal import Decimal

from sqlalchemy import text

# get_engine() живёт в вашем репозитории, в src/prime/config.py.
# Она читает строку подключения из .env — поэтому пароля здесь нет.
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def query(sql: str, **params) -> list[dict]:
    """Выполнить SQL-запрос и вернуть строки списком словарей {колонка: значение}."""
    try:
        with engine.connect() as conn:
            # Моменты времени в базе хранятся с часовым поясом. День события считаем
            # по UTC — так он у всех одинаковый, какие бы настройки ни стояли у вас.
            conn.execute(text("set time zone 'UTC'"))
            result = conn.execute(text(sql), params)
            return [dict(row) for row in result.mappings()]
    finally:
        # Соединение не держим: сервер у всей группы общий, а после сна ноутбука
        # старое соединение всё равно мёртвое.
        engine.dispose()


print("подключились к базе как:", query("select current_user as login")[0]["login"])

подключились к базе как: atkutyavina


### 1.3 Строка участка

Показывает вашу строку из `prime.assignments` — ту же, что на первом семинаре. Чужих строк база не показывает. Скопируйте напечатанное в ячейку 1.1 и запустите 1.1 ещё раз.

За то, что в 1.1 стоят именно ваши значения, отвечаете вы: код с чужим участком спокойно всё посчитает, только не то.

In [3]:
mine = query("select * from prime.assignments where login = current_user")

if mine:
    row = mine[0]
    print("Ваш участок. Скопируйте в ячейку 1.1 и запустите её:\n")
    print(f"TABLE = {row['table_name']!r}")
    print(f"SEGMENT_COLUMN = {row['segment_column']!r}")
    print(f"SEGMENT = {row['segment']!r}")
    print(f"PERIOD_START = {str(row['period_start'])!r}")
    print(f"PERIOD_END = {str(row['period_end'])!r}")
else:
    # Так бывает, когда ноутбук запускает преподаватель под своей ролью.
    print("Строки участка для этого пользователя в базе нет.")

Ваш участок. Скопируйте в ячейку 1.1 и запустите её:

TABLE = 'service_usage'
SEGMENT_COLUMN = 'device_type'
SEGMENT = 'desktop'
PERIOD_START = '2025-04-01'
PERIOD_END = '2025-04-30'


### 1.4 Запрос

Четыре таблицы устроены по-разному, но для задач нужно одно и то же. Поэтому запрос приводит любую из них к **пяти полям**:

| Поле | Что это |
|---|---|
| `row_id` | идентификатор строки |
| `actor_id` | участник — тот, чьи это строки |
| `day` | день события, тип `datetime.date` |
| `ok` | состоялось ли событие: `True` или `False` |
| `value` | сколько: деньги, минуты или часы, тип `Decimal` |

Что это значит в **вашей** таблице:

| Таблица | Одна строка — это | Участник | Сегмент | `ok = True` | `ok = False` | `value` |
|---|---|---|---|---|---|---|
| `payments` | попытка списать плату за подписку | подписка | способ оплаты: `card`, `sbp`, `wallet`, `balance` | списание прошло | не прошло или деньги вернули | сумма списания, ₽ |
| `transactions` | покупка клиента в партнёрском сервисе | клиент | канал: `app` — приложение, `web` — сайт, `pos` — касса | покупка прошла | отклонена или проведена и отменена | сумма покупки, ₽; бывает ноль и меньше нуля |
| `service_usage` | сессия в сервисе ПРАЙМ | клиент | устройство | сессия записана корректно: конец позже начала | конец не позже начала | длительность, минуты |
| `support_tickets` | обращение в поддержку | клиент | тема обращения | обращение решено | ещё в работе | время от создания до решения, часы; у нерешённых — `None` |

**Участник** здесь — просто тот, чьи это строки. Не путайте с участниками подписки из `subscription_members`, о которых шла речь на первом семинаре. А у `support_tickets` сумма `value` по дню растёт и от числа обращений, и от медленных решений — это не одно и то же.

Как работает запрос:

1. Блок `acts` выбирает **1000 участников** вашего сегмента за ваше окно. Это выборка, а не весь сегмент: поэтому строк в `rows` меньше, чем вы насчитали в хвосте первого семинара, — там был весь срез. Подгонять под то число не нужно. В ответах словами пишите «в выгрузке» или «среди 1000 подписок / клиентов», а не «по всем картам за квартал». Перемешивание через `md5` делает выбор похожим на случайный, но при каждом запуске — одинаковым.
2. Основной запрос берёт строки этих участников — **только в вашем сегменте и в вашем окне** (тот же фильтр, что в `acts`) — и переименовывает колонки в пять полей.
3. В `{фигурных скобках}` — названия колонок из словаря `COLUMNS`: они подставляются под вашу таблицу. После двоеточия (`:segment`, `:period_start`, `:period_end`) — ваши значения из ячейки 1.1.
4. Граница окна: момент события `>=` первого дня и `<` дня, следующего за последним. Так последний день попадает в окно целиком.

In [4]:
# Как в каждой из четырёх таблиц называются колонки, из которых получаются пять полей.
COLUMNS = {
    "payments": {
        "segment_column": "payment_method",   # колонка сегмента
        "row_id": "payment_id",                # -> row_id
        "actor": "subscription_id",            # -> actor_id
        "moment": "paid_at",                   # момент события -> day
        "ok": "status = 'success'",            # -> ok
        "value": "amount",                     # -> value
    },
    "transactions": {
        "segment_column": "channel",
        "row_id": "transaction_id",
        "actor": "client_id",
        "moment": "occurred_at",
        "ok": "status = 'success'",
        "value": "amount",
    },
    "service_usage": {
        "segment_column": "device_type",
        "row_id": "usage_id",
        "actor": "client_id",
        "moment": "started_at",
        "ok": "ended_at > started_at",
        "value": "extract(epoch from (ended_at - started_at)) / 60.0",      # секунды -> минуты
    },
    "support_tickets": {
        "segment_column": "category",
        "row_id": "ticket_id",
        "actor": "client_id",
        "moment": "created_at",
        "ok": "resolved_at is not null",
        "value": "extract(epoch from (resolved_at - created_at)) / 3600.0",  # секунды -> часы
    },
}

SLICE_SQL = """
with acts as (
    select actor_id
    from (
        select distinct {actor}::text as actor_id
        from prime.{table}
        where {segment_column} = :segment
          and {moment} >= cast(:period_start as date)
          and {moment} <  cast(:period_end as date) + 1
    ) s
    order by md5(actor_id)
    limit 1000
)
select t.{row_id}::text as row_id,
       t.{actor}::text  as actor_id,
       t.{moment}::date as day,
       {ok}             as ok,
       {value}          as value
from prime.{table} t
join acts a on a.actor_id = t.{actor}::text
where t.{segment_column} = :segment
  and t.{moment} >= cast(:period_start as date)
  and t.{moment} <  cast(:period_end as date) + 1
order by t.{row_id}
"""

print("запрос описан")

запрос описан


### 1.5 Выгрузка

Достаём участок из базы в переменную `rows` — список словарей, по одному на строку. **Все задачи ниже работают с `rows`.**

Сколько в выгрузке строк, участников и дней — ячейка не печатает: это вы посчитаете сами в задаче 1.

In [5]:
# В текст запроса подставляются только известные названия таблиц и колонок —
# поэтому сначала проверяем, что значения в ячейке 1.1 из этого списка.
if TABLE not in COLUMNS:
    raise ValueError(f"TABLE должна быть одной из {list(COLUMNS)}, сейчас {TABLE!r}")
if SEGMENT_COLUMN != COLUMNS[TABLE]["segment_column"]:
    raise ValueError(f"у таблицы {TABLE} колонка сегмента называется {COLUMNS[TABLE]['segment_column']!r}")
try:
    window_days = (date.fromisoformat(PERIOD_END) - date.fromisoformat(PERIOD_START)).days + 1
except (TypeError, ValueError):
    raise ValueError("PERIOD_START и PERIOD_END — строки вида '2025-04-01'") from None

# Собираем запрос под вашу таблицу и выполняем. Обычно несколько секунд;
# если висит дольше минуты — см. «Если что-то не работает» в тексте задания.
slice_sql = SLICE_SQL.format(table=TABLE, **COLUMNS[TABLE])
rows = query(slice_sql, segment=SEGMENT, period_start=PERIOD_START, period_end=PERIOD_END)

# Защита от ошибок в значениях 1.1: пустая выгрузка или дни за пределами окна.
if not rows:
    raise ValueError("выгрузка пустая — проверьте значения в ячейке 1.1")
if len({row["day"] for row in rows}) > window_days:
    raise ValueError("в выгрузке дней больше, чем в окне — граница периода съехала")

print("Выгрузка готова. Так выглядят первые три строки:")
rows[:3]

Выгрузка готова. Так выглядят первые три строки:


[{'row_id': '004a29bd-f913-4e68-be1f-876480fd4fff',
  'actor_id': '26fc280c-5a45-48dd-aeec-80ca7d0a3422',
  'day': datetime.date(2025, 4, 20),
  'ok': True,
  'value': Decimal('11.7500000000000000')},
 {'row_id': '0097c7d1-7095-4a49-902d-2acc263c8e8d',
  'actor_id': 'd5604e0c-0d7a-40cf-a35b-1071e8773d2c',
  'day': datetime.date(2025, 4, 17),
  'ok': True,
  'value': Decimal('21.3000000000000000')},
 {'row_id': '00aaf0cd-7000-4fd4-8c3b-ee4db4e4f834',
  'actor_id': 'be962bf7-6be2-4f30-accf-ab0d4f5bdf8c',
  'day': datetime.date(2025, 4, 4),
  'ok': True,
  'value': Decimal('17.5333333333333333')}]

### 1.6 Формат ответов

Служебная функция: по ней итог в части 3 проверяет, что ответ записан в нужном формате — список, число, дата строкой. **Верно ли посчитано, она не знает и не сообщает** — это проверяет преподаватель. Код ячейки свёрнут — так задумано, просто запустите её.

In [6]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "list из 4 элементов: [int, int, float до 2 знаков, int]",
    2: "int",
    3: "dict: ключи из Mon…Sun, значения float до 2 знаков",
    4: "list [str, int]",
    5: "list из 3 элементов, каждый — list ['ГГГГ-ММ-ДД', float до 2 знаков]",
    6: "list [int, 'ГГГГ-ММ-ДД']",
    7: "list [float до 2 знаков, int]",
    8: "float до 2 знаков",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def money(x) -> bool:  # float, у которого не больше двух знаков после запятой
        return type(x) is float and round(x, 2) == x

    def iso_day(x) -> bool:  # дата строкой 'ГГГГ-ММ-ДД'
        try:
            return type(x) is str and date.fromisoformat(x).isoformat() == x
        except ValueError:
            return False

    if answer is None:
        return "не решена: функция вернула None — в ней остался ... или забыт return"
    checks = {
        1: lambda a: type(a) is list and len(a) == 4 and type(a[0]) is int and type(a[1]) is int
                     and money(a[2]) and type(a[3]) is int,
        2: lambda a: type(a) is int,
        3: lambda a: type(a) is dict and set(a) <= {"Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"}
                     and all(money(v) for v in a.values()),
        4: lambda a: type(a) is list and len(a) == 2 and type(a[0]) is str and type(a[1]) is int,
        5: lambda a: type(a) is list and len(a) == 3
                     and all(type(p) is list and len(p) == 2 and iso_day(p[0]) and money(p[1]) for p in a),
        6: lambda a: type(a) is list and len(a) == 2 and type(a[0]) is int and iso_day(a[1]),
        7: lambda a: type(a) is list and len(a) == 2 and money(a[0]) and type(a[1]) is int,
        8: lambda a: money(a),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r}"


print("проверка формата готова")

проверка формата готова


## Часть 2. Восемь задач

У каждой задачи два шага:

1. **Посчитать.** Допишите функцию `taskN(rows)` — она должна вернуть ответ ровно в указанном формате. Имена функций и переменных `answer_N` не меняйте: по ним работа проверяется автоматически, в том числе на другом участке.
2. **Ответить на вопрос.** Для ответа почти всегда нужно досчитать одно-два числа — для этого под вопросом есть пустая ячейка. Сам ответ пишется в ячейке «✍️ Ответ на вопрос N»: дважды щёлкните по ней, **заголовок не трогайте**, замените строку курсивом своим текстом и нажмите `Shift + Enter`.

**Ячейки расчётов тоже выполняются на контрольном участке.** Считайте в них от `rows` и `answer_N`; конкретные даты, `actor_id` и сегмент руками не вписывайте — на другом участке их нет, и ячейка упадёт.

**Не получилась задача — не оставляйте падающий код.** Ячейка с ошибкой останавливает Run All, и ноутбук не проходит целиком — это минус баллы за воспроизводимость. Верните в тело функции `...`: она вернёт `None`, итог покажет «❌ не решена», а остальное проверится как обычно.

**Про ответы словами.** Код можно писать вместе с ИИ-ассистентом — это нормально, отметьте это в декларации. А ответы пишите сами: здесь оценивается, как думаете вы. **Текст, написанный LLM, получает 0 баллов.** Хороший ответ — 2–4 предложения, в них ваши числа и ваш вывод. Сомнительный вывод баллов не приносит, даже если числа верные.

**Ноутбук не скажет, правильно ли вы посчитали.** Итог в части 3 проверяет только формат. Правильность проверяет преподаватель — на вашем участке и на контрольном.

### Три особенности данных, на которых спотыкаются чаще всего

* **`value` приходит типом `Decimal`, а не `float`.** Это точный тип для денег. `Decimal` складывается с `Decimal` и с целыми числами, а с `float` — нет: `Decimal("1.5") + 0.5` падает с `TypeError`. В ноутбуке С2-идиомы суммы копились от `total = 0.0` — здесь так не выйдет. Копите суммы в `Decimal` (`defaultdict(Decimal)`, `sum(...)` без стартового `0.0`), а к `float` приводите в самом конце: `float(round(сумма, 2))`.
* **В `support_tickets` у нерешённых обращений `value` равно `None`** — времени до решения ещё нет. Проверяйте `row["value"] is not None` там, где складываете `value`, а не `if row["value"]`: второе молча выкинет и честный ноль. Сами строки с `None` из `rows` не удаляйте — это нерешённые обращения, и они нужны в задачах 1, 2 и в вопросе 1.
* **Строки с `ok = True` и отрицательным или нулевым `value` не выбрасывайте.** В `transactions` такие есть — например, покупка на ноль рублей. Считайте их как есть.

Нужные приёмы разобраны в [ноутбуке С2-идиомы](https://github.com/tikhomirovd/python-for-ba-hse-2026/blob/master/02-среда-git-python/семинар/С2-идиомы.ipynb): генераторные выражения, множества, `defaultdict` и `Counter`, `sorted` с ключом, `enumerate`, `try/except`.

### Задача 1. Паспорт участка

Руководитель начинает каждую неделю с одного вопроса: «Сколько у нас всего — и сколько из этого настоящего?» Прежде чем что-то анализировать, надо знать масштаб участка.

**Что посчитать**

Четыре числа по вашей выгрузке `rows`:

1. сколько всего строк;
2. сколько строк с `ok = True` — событий, которые состоялись;
3. сколько `value` набежало по строкам с `ok = True` — выручка, минуты или часы, смотря какая у вас таблица;
4. сколько **различных** дней встречается в выгрузке — по всем строкам, а не только по состоявшимся.

**Формат ответа:** `[всего, с_ok, сумма, дней]` — например `[2222, 2100, 123456.7, 17]`: целое, целое, `float` до двух знаков, целое.

**Подсказка:** `len(rows)`; `sum(1 for row in rows if ...)`; множество дней `{row["day"] for row in rows}`.

In [7]:
def task1(rows: list[dict]) -> list:
    # верните [всего, с_ok, сумма, дней]
    total = len(rows)
    ok_count = sum(1 for row in rows if row["ok"])
    total_value = float(round(sum(row["value"] for row in rows if row["ok"]), 2))
    distinct_days = len({row["day"] for row in rows})

    return [total, ok_count, total_value, distinct_days]


answer_1 = task1(rows)
answer_1

[1317, 1315, 36010.43, 30]

**Вопрос 1.** **Что не случилось.** Какая доля строк вашей выгрузки не состоялась (`ok = False`)? Что это значит в жизни вашей таблицы — что именно не произошло? Сколько `value` приходится на эти строки — или объясните, почему у вашей таблицы это число не имеет смысла или его нельзя посчитать.

In [8]:
# Числа для ответа на вопрос 1 считайте здесь — по rows и answer_1.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
total, ok_count, _, _ = answer_1
share_not_ok = (total - ok_count) / total

not_ok_values = [row["value"] for row in rows if not row["ok"]]
has_none = any(v is None for v in not_ok_values)

print("Доля несостоявшихся строк:", round(share_not_ok, 4))
print("Есть ли None среди value у несостоявшихся строк:", has_none)

if has_none:
    known = [v for v in not_ok_values if v is not None]
    print("Сумма value по несостоявшимся строкам, где value известен:", float(round(sum(known), 2)) if known else 0.0,
          "- штук без value:", sum(1 for v in not_ok_values if v is None))
else:
    print("Сумма value по несостоявшимся строкам:", float(round(sum(not_ok_values), 2)) if not_ok_values else 0.0)

not_ok_rows = [row for row in rows if not row["ok"]]

zero_duration = [row for row in not_ok_rows if row["value"] == 0]
negative_duration = [row for row in not_ok_rows if row["value"] < 0]

Доля несостоявшихся строк: 0.0015
Есть ли None среди value у несостоявшихся строк: False
Сумма value по несостоявшимся строкам: -71.85


In [9]:
print("Несостоявшихся строк всего:", len(not_ok_rows))
print("Из них с value == 0:", len(zero_duration))
print("Из них с value < 0:", len(negative_duration))

print("\nОтрицательные строки:")
for row in negative_duration:
    print(row)

Несостоявшихся строк всего: 2
Из них с value == 0: 0
Из них с value < 0: 2

Отрицательные строки:
{'row_id': '33576a7f-abbd-4246-8fac-cce6a91478bf', 'actor_id': '2327d584-9aea-4434-914a-e600769ee253', 'day': datetime.date(2025, 4, 2), 'ok': False, 'value': Decimal('-54.2500000000000000')}
{'row_id': '654e59e6-23a4-4375-a783-c1becccccb48', 'actor_id': 'd971088f-f2d0-4d6f-85f2-09b87122084b', 'day': datetime.date(2025, 4, 5), 'ok': False, 'value': Decimal('-17.6000000000000000')}


In [10]:
for row in negative_duration:
    same_actor_same_day = [r for r in rows if r["actor_id"] == row["actor_id"] and r["day"] == row["day"]]
    print(row["actor_id"], row["day"], "-> строк в этот день у участника:", len(same_actor_same_day))

2327d584-9aea-4434-914a-e600769ee253 2025-04-02 -> строк в этот день у участника: 1
d971088f-f2d0-4d6f-85f2-09b87122084b 2025-04-05 -> строк в этот день у участника: 1


#### ✍️ Ответ на вопрос 1

_2 строки в моей таблице не состоялись (0,15%), это значит, что в 2 строках время начала сессии позже времени окончания. Причём, в обеих строках при этом стоит отрицательное время сессии (суммарно -71,85 минуты), но других сессий в этот день у пользователей нет, то есть путаницы между сессиями не возникло, но время начала и окончания могли быть перепутаны. Отрицательное время по сути не несёт смысла как метрика, а является следствием ошибки в записи начала и конца сессии._

### Задача 2. И получилось, и нет

Самые интересные участники — те, у кого в одном окне было и «получилось», и «не получилось»: оплата сначала не прошла, а потом прошла; одну покупку отклонили, другую провели. Руководитель хочет знать, сколько таких.

**Что посчитать**

Сколько **различных** участников (`actor_id`) имеют в выгрузке хотя бы одну строку с `ok = True` **и** хотя бы одну строку с `ok = False`.

**Формат ответа:** целое число, например `123`. Ноль — тоже законный ответ.

**Подсказка:** два множества участников и операция `&` между ними.

In [11]:
def task2(rows: list[dict]) -> int:
    # верните число участников
    ok_actors = {row["actor_id"] for row in rows if row["ok"]}
    not_ok_actors = {row["actor_id"] for row in rows if not row["ok"]}
    both = ok_actors & not_ok_actors
    return len(both)


answer_2 = task2(rows)
answer_2

0

**Вопрос 2.** **Почему столько.** Сколько в среднем строк приходится на одного участника в вашей выгрузке? Объясните, почему ответ задачи 2 получился именно таким. Если это ноль или единица — это не ошибка: объясните, откуда он берётся. Подумайте, к чему относится сегмент: к отдельной строке (у одного участника строки могут быть с разными значениями) или к участнику целиком (у всех его строк оно одно).

In [12]:
# Числа для ответа на вопрос 2 считайте здесь — по rows и answer_2.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
distinct_actors = len({row["actor_id"] for row in rows})
avg_rows_per_actor = len(rows) / distinct_actors

print("Различных участников:", distinct_actors)
print("Строк в выгрузке:", len(rows))
print("В среднем строк на участника:", round(avg_rows_per_actor, 3))

Различных участников: 1000
Строк в выгрузке: 1317
В среднем строк на участника: 1.317


In [13]:
from collections import Counter

rows_per_actor = Counter(row["actor_id"] for row in rows)
print("Максимум строк у одного участника:", max(rows_per_actor.values()))
print("Сколько участников имеют больше 1 строки:", sum(1 for c in rows_per_actor.values() if c > 1))
print("Распределение (сколько строк -> сколько участников с таким числом):", Counter(rows_per_actor.values()))

Максимум строк у одного участника: 6
Сколько участников имеют больше 1 строки: 230
Распределение (сколько строк -> сколько участников с таким числом): Counter({1: 770, 2: 173, 3: 35, 4: 16, 5: 4, 6: 2})


#### ✍️ Ответ на вопрос 2

_На 1 участника в среднем приходится 1,317 строки. То есть большинство участников заходили в ПРАЙМ с десктопной версии 1 раз за месяц - это вполне адекватное значение, тк они могли заходить и с других устройств. Участников с несостоявшимися сессиями всего 2 (маленькая выборка), и у них обоих только по одной строке в выгрузке, поэтому ответ в задаче 2 - 0. Сегмент десктопа в данном случае относится к строке, а не к участнику, тк каждый участник мог заходить и другими способами._

### Задача 3. Главный день недели

Команда решает, в какой день недели ставить дежурство и запускать рассылки. Нужна раскладка участка по дням недели.

**Что посчитать**

Сумма `value` по строкам с `ok = True` — отдельно для каждого дня недели.

**Формат ответа:** словарь `{'Mon': 12345.67, 'Tue': ...}` — только дни недели, которые встретились; суммы `float` до двух знаков. Название дня берите из списка `WEEKDAYS` по номеру `row["day"].weekday()` (понедельник — 0).

**Подсказка:** `defaultdict(Decimal)`. Почему не `day.strftime('%a')`: результат зависит от языковых настроек — стоит где-нибудь вызвать `locale.setlocale`, и вместо `'Tue'` получится `'вт'`, а ответ разойдётся с форматом.

In [14]:
WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]  # WEEKDAYS[0] — понедельник


def task3(rows: list[dict]) -> dict:
    # верните {'Mon': сумма, 'Tue': сумма, ...}
    sums = defaultdict(Decimal)
    for row in rows:
        if row["ok"]:
            sums[WEEKDAYS[row["day"].weekday()]] += row["value"]

    return {day: float(round(sums[day], 2)) for day in WEEKDAYS if day in sums}


answer_3 = task3(rows)
answer_3

{'Mon': 4664.1,
 'Tue': 5598.15,
 'Wed': 6900.77,
 'Thu': 4357.58,
 'Fri': 5424.23,
 'Sat': 4518.13,
 'Sun': 4547.47}

**Вопрос 3.** **Сумма или среднее.** Сколько **различных дат** вашей выгрузки приходится на каждый день недели? Считайте даты, а не строки. Какой день лидирует по сумме из задачи 3, а какой — по средней сумме на одну такую дату (сумма из задачи 3, делённая на число дат этого дня недели)? Если лидеры разные — какой ответ вы понесли бы руководителю и почему; если одинаковые — объясните, почему так вышло.

In [15]:
# Числа для ответа на вопрос 3 считайте здесь — по rows и answer_3.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
dates_by_weekday = defaultdict(set)
for row in rows:
    dates_by_weekday[WEEKDAYS[row["day"].weekday()]].add(row["day"])

dates_count = {day: len(dates_by_weekday[day]) for day in WEEKDAYS if day in dates_by_weekday}

leader_by_sum = max(answer_3, key=answer_3.get)

avg_per_date = {day: answer_3[day] / dates_count[day] for day in answer_3}
leader_by_avg = max(avg_per_date, key=avg_per_date.get)

print("Дат по дням недели:", dates_count)
print("Сумма по дням недели (задача 3):", answer_3)
print("Средняя сумма на дату по дням недели:", {d: round(v, 2) for d, v in avg_per_date.items()})
print("Лидер по сумме:", leader_by_sum, "\nЛидер по средней на дату:", leader_by_avg)

Дат по дням недели: {'Mon': 4, 'Tue': 5, 'Wed': 5, 'Thu': 4, 'Fri': 4, 'Sat': 4, 'Sun': 4}
Сумма по дням недели (задача 3): {'Mon': 4664.1, 'Tue': 5598.15, 'Wed': 6900.77, 'Thu': 4357.58, 'Fri': 5424.23, 'Sat': 4518.13, 'Sun': 4547.47}
Средняя сумма на дату по дням недели: {'Mon': 1166.03, 'Tue': 1119.63, 'Wed': 1380.15, 'Thu': 1089.39, 'Fri': 1356.06, 'Sat': 1129.53, 'Sun': 1136.87}
Лидер по сумме: Wed 
Лидер по средней на дату: Wed


In [16]:
wed_days = sorted(row["day"] for row in rows if row["ok"] and row["day"].weekday() == 2)
wed_sums = defaultdict(Decimal)
for row in rows:
    if row["ok"] and row["day"].weekday() == 2:
        wed_sums[row["day"]] += row["value"]

for day in sorted(wed_sums):
    print(day, float(round(wed_sums[day], 2)))

2025-04-02 1625.1
2025-04-09 1317.63
2025-04-16 1635.98
2025-04-23 982.68
2025-04-30 1339.37


In [17]:
count_ok_by_weekday = defaultdict(int)
for row in rows:
    if row["ok"]:
        count_ok_by_weekday[WEEKDAYS[row["day"].weekday()]] += 1

for day in WEEKDAYS:
    if day in count_ok_by_weekday:
        avg_per_session = answer_3[day] / count_ok_by_weekday[day]
        print(day, " сессий:", count_ok_by_weekday[day], ", среднее на сессию:", round(avg_per_session, 2))

Mon  сессий: 161 , среднее на сессию: 28.97
Tue  сессий: 209 , среднее на сессию: 26.79
Wed  сессий: 246 , среднее на сессию: 28.05
Thu  сессий: 167 , среднее на сессию: 26.09
Fri  сессий: 196 , среднее на сессию: 27.67
Sat  сессий: 172 , среднее на сессию: 26.27
Sun  сессий: 164 , среднее на сессию: 27.73


#### ✍️ Ответ на вопрос 3

_Вторников и сред по 5, остальных дней недели по 4. И по сумме, и по средней сумме лидирует среда. Так выходит, потому что в среду зафиксировано наибольшее количество сессий, и это позволяет среде стать лидером, несмотря на то, что по средней длительности сессии она уступает понедельнику. Но лидерство при этом устойчиво, тк сессий по средам больше сессий по вторникам (второе место) на 17,7%, а средние длительности сессий между днями недели несильно отличаются._

### Задача 4. Самый активный участник

Маркетинг хочет наградить самого активного участника участка — того, у кого больше всего состоявшихся событий. Приз уже куплен.

**Что посчитать**

Участник с наибольшим числом строк с `ok = True` и это число. Если таких несколько — тот, чей `actor_id` меньше при сравнении как текст (так сравнивает `min`).

**Формат ответа:** `[actor_id, число строк]` — например `['aaaaaaaa-1111-2222-3333-bbbbbbbbbbbb', 9]`: текст (`str`) и целое.

**Подсказка:** `Counter` по строкам с `ok = True` считает, сколько состоявшихся событий у каждого участника; `max` находит наибольшее число, `min` — наименьший `actor_id` среди тех, у кого оно такое.

In [18]:
def task4(rows: list[dict]) -> list:
    # верните [actor_id, число строк]
    counts = Counter(row["actor_id"] for row in rows if row["ok"])

    max_count = max(counts.values())
    leaders = [actor for actor, c in counts.items() if c == max_count]
    winner = min(leaders)

    return [winner, max_count]


answer_4 = task4(rows)
answer_4

['3581c3e6-208a-44b8-b1e2-8e60caa613e4', 6]

**Вопрос 4.** **Кто победил на самом деле.** Сколько участников делят первое место? Если больше одного — что на самом деле выбрало вашего «победителя»: данные или правило? И что в вашей таблице вообще значит «самый активный» — стоит ли маркетингу вручать приз по такому рейтингу?

In [19]:
# Числа для ответа на вопрос 4 считайте здесь — по rows и answer_4.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
counts = Counter(row["actor_id"] for row in rows if row["ok"])
max_count = answer_4[1]
leaders = [actor for actor, c in counts.items() if c == max_count]

print("Максимум состоявшихся строк у одного участника:", max_count)
print("Сколько участников достигли этого максимума:", len(leaders))
print("actor_id-победители:", leaders)

Максимум состоявшихся строк у одного участника: 6
Сколько участников достигли этого максимума: 2
actor_id-победители: ['f01b3a91-1878-4a61-b84d-bec663dab43b', '3581c3e6-208a-44b8-b1e2-8e60caa613e4']


In [20]:
for actor in leaders:
    total_value = sum(row["value"] for row in rows if row["actor_id"] == actor and row["ok"])
    print(actor, "  сессий:", max_count, "  суммарно минут:", float(round(total_value, 2)))

f01b3a91-1878-4a61-b84d-bec663dab43b   сессий: 6   суммарно минут: 103.08
3581c3e6-208a-44b8-b1e2-8e60caa613e4   сессий: 6   суммарно минут: 113.2


#### ✍️ Ответ на вопрос 4

_2 участника делят первое место. Победителя выбрало правило, но при этом он является лидером и по суммарному времени сессий, но это случайность. А самый активный в моей таблице - пользователь с максимальным количеством сессий с десктопа, но не активностью в целом, поэтому приз не стоило бы вручать._

### Задача 5. Три пиковых дня

Чтобы планировать нагрузку на команду и серверы, нужно знать пиковые дни окна.

**Что посчитать**

Три даты с наибольшей суммой `value` по строкам с `ok = True` — по убыванию суммы. Если суммы равны, раньше идёт более ранняя дата.

**Формат ответа:** `[['2024-03-15', 98765.43], ['2024-03-02', 91200.1], ['2024-03-21', 90000.0]]` — дата строкой `'ГГГГ-ММ-ДД'`, сумма `float` до двух знаков.

**Подсказка:** сначала словарь «день → сумма», потом `sorted(..., key=...)`: ключ-кортеж `(-сумма, день)` сортирует по убыванию суммы, а при равенстве — по возрастанию даты. Дата строкой — `day.isoformat()`.

In [21]:
def task5(rows: list[dict]) -> list:
    # верните [['ГГГГ-ММ-ДД', сумма], [...], [...]]
    day_sums = defaultdict(Decimal)
    for row in rows:
        if row["ok"]:
            day_sums[row["day"]] += row["value"]

    ranked = sorted(day_sums.items(), key=lambda kv: (-kv[1], kv[0]))

    top3 = ranked[:3]
    return [[day.isoformat(), float(round(total, 2))] for day, total in top3]


answer_5 = task5(rows)
answer_5

[['2025-04-11', 1724.82], ['2025-04-25', 1636.97], ['2025-04-16', 1635.98]]

**Вопрос 5.** **Откуда пик.** Какие это дни недели, сколько среди них суббот и воскресений? Сформулируйте одну гипотезу, почему пик пришёлся на эти даты, и напишите, какой расчёт на данных её подтвердил бы или опроверг.

In [22]:
# Числа для ответа на вопрос 5 считайте здесь — по rows и answer_5.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
for day_str, value in answer_5:
    d = date.fromisoformat(day_str)
    print(day_str, "->", WEEKDAYS[d.weekday()])

weekend_count = sum(1 for day_str, _ in answer_5 if date.fromisoformat(day_str).weekday() >= 5)
print("Из 3 пиковых дней приходится на сб/вс:", weekend_count)

2025-04-11 -> Fri
2025-04-25 -> Fri
2025-04-16 -> Wed
Из 3 пиковых дней приходится на сб/вс: 0


In [23]:
weekday_total = sum(answer_3[d] for d in ["Mon","Tue","Wed","Thu","Fri"] if d in answer_3)
weekend_total = sum(answer_3[d] for d in ["Sat","Sun"] if d in answer_3)
print("Будни:", float(weekday_total), ", выходные:", float(weekend_total))

Будни: 26944.83 , выходные: 9065.6


In [24]:
fri_sums = defaultdict(Decimal)
for row in rows:
    if row["ok"] and row["day"].weekday() == 4:
        fri_sums[row["day"]] += row["value"]
for day in sorted(fri_sums):
    sessions = sum(1 for row in rows if row["ok"] and row["day"] == day)
    print(day, "Сумма:", float(round(fri_sums[day], 2)), ", сессий:", sessions)

2025-04-04 Сумма: 1023.07 , сессий: 41
2025-04-11 Сумма: 1724.82 , сессий: 57
2025-04-18 Сумма: 1039.38 , сессий: 47
2025-04-25 Сумма: 1636.97 , сессий: 51


#### ✍️ Ответ на вопрос 5

_Первые 2 места - пятницы, третье - среда, суббот и воскресений нет. Выходных здесь нет, скорее опять же, из-за сегмента десктопа, именно в эти дни он может пользоваться меньшей популярностью, тк удобнее пользоваться подпиской другими способами. Но среда при этом вполне логично присутствует в топе, тк в целом занимает первое место по итогам 3 задания. Гипотеза касательно пятниц: с учётом того, что ранее в 3 задании пятница не вошла в топ, можно предположить, что важен не столько день недели, сколько конкретные даты, а 11 и 25 апреля - могут быть близки к дням зарплаты, и поэтому вести к доп пользованиям сервисами. Расчёт суммарных длительностей и количества сессий по пятницам может показать, что дело не в дне недели, а влияние дат выплат зарплат можно проверить путём определения пиковых дат в другие месяцы._

### Задача 6. Когда набралась половина

Финансам важно, равномерно ли набирается сумма внутри окна: к какому дню набегает половина суммы по состоявшимся событиям.

**Что посчитать**

1. Возьмите дни, в которых есть хотя бы одна строка с `ok = True`, и расставьте по возрастанию даты.
2. Идите по ним и копите сумму `value` по строкам с `ok = True`.
3. Найдите первый день, на котором накопленное **достигло половины** суммы из пункта 3 задачи 1 (всё `value` по строкам с `ok = True`) или превысило её, и его порядковый номер в списке дней из шага 1, считая с 1.

**Формат ответа:** `[номер, 'ГГГГ-ММ-ДД']` — например `[12, '2024-03-15']`.

**Подсказка:** `sums` — словарь «день → сумма», как в задаче 5. `for number, day in enumerate(sorted(sums), start=1):` — `enumerate` сам считает номер.

In [25]:
def task6(rows: list[dict]) -> list:
    # верните [номер дня, 'ГГГГ-ММ-ДД']
    day_sums = defaultdict(Decimal)
    for row in rows:
        if row["ok"]:
            day_sums[row["day"]] += row["value"]

    ordered_days = sorted(day_sums)

    total = sum(day_sums.values())
    target = total / 2

    cumulative = Decimal(0)
    for number, day in enumerate(ordered_days, start=1):
        cumulative += day_sums[day]
        if cumulative >= target:
            return [number, day.isoformat()]

    raise RuntimeError("Не нашлось дня, где накопилась половина")


answer_6 = task6(rows)
answer_6

[16, '2025-04-16']

**Вопрос 6.** **Равномерно ли.** Какую долю дней заняла первая половина суммы: номер дня из задачи 6, делённый на число дней в списке из шага 1? Номер дня целый, поэтому при равномерном накоплении получилось бы около половины, округлённой вверх (15 из 30, 16 из 31) — сравнивайте с этим, а не с ровными 50 %. У вас раньше или позже — и что это говорит о том, как менялся участок внутри окна? Если вышло у середины — так и напишите.

In [26]:
# Числа для ответа на вопрос 6 считайте здесь — по rows и answer_6.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
ok_days = sorted({row["day"] for row in rows if row["ok"]})
n_days = len(ok_days)
share_days = answer_6[0] / n_days

import math
even_reference = math.ceil(n_days / 2) / n_days

print("Всего дней с ok=True:", n_days)
print("Номер дня половины (задача 6):", answer_6[0])
print("Доля дней:", round(share_days, 3), ", ориентир при равномерном накоплении:", round(even_reference, 3))

Всего дней с ok=True: 30
Номер дня половины (задача 6): 16
Доля дней: 0.533 , ориентир при равномерном накоплении: 0.5


In [27]:
day_sums = defaultdict(Decimal)
for row in rows:
    if row["ok"]:
        day_sums[row["day"]] += row["value"]

ordered = sorted(day_sums)
total = sum(day_sums.values())

cumulative = Decimal(0)
for number, day in enumerate(ordered, start=1):
    cumulative += day_sums[day]
    if number in (14, 15, 16, 17):
        print(day, "накоплено:", round(float(cumulative / total * 100), 2), "%")

first_half = sum(day_sums[d] for d in ordered[:15])
print("Сумма первых 15 дней:", float(round(first_half, 2)),
      ", последних 15:", float(round(total - first_half, 2)))

2025-04-14 накоплено: 46.75 %
2025-04-15 накоплено: 48.47 %
2025-04-16 накоплено: 53.01 %
2025-04-17 накоплено: 56.41 %
Сумма первых 15 дней: 17454.38 , последних 15: 18556.05


#### ✍️ Ответ на вопрос 6

_16 дней из 30 составляют 53.3%, день почти у середины (на день позже), то есть накопление происходит практически равномерно, но вторая половина месяца немного перевешивает (примерно на 3%). Важно отметить, что 16.04 входило в топ-3 дат по суммарной длительности, поэтому именно этот день сделал сильное прибавление к накопленной длительности, но при этом 15ому дню до 50% не хватало примерно 1,5%, а значит, даже если бы 16.04 был более среднестатистическим днём, переход через 50% всё равно бы произошёл._

### Задача 7. Функция, которая не падает

Соседний отдел присылает суммы строками — так, как их набрал оператор: с запятой, с пробелами, иногда с прочерком. Нужна функция, которая превращает такую строку в число и не роняет ежедневный отчёт. Данные в этой задаче у всех одинаковые — список `RAW`.

**Что посчитать**

1. Функция `to_value(x)`: получает строку (или `None`) и возвращает `float` — или `None`, если число разобрать нельзя. Падать нельзя ни на одном значении. Правила записи:
   * точка и запятая — десятичный разделитель: `'89.10'` → `89.1`, `'349,90'` → `349.9`;
   * пробел и неразрывный пробел внутри числа — разделители тысяч: `'1 499,50'` → `1499.5`;
   * пробелы по краям ничего не значат;
   * прочерк `'—'` и пустая строка — не число: `None`, а не `0.0`.
2. По списку `RAW`: сумма всего, что разобралось, и сколько значений отброшено. `None` тоже считается отброшенным.

**Формат ответа:** `[сумма, отброшено]` — `float` до двух знаков и целое.

**Подсказка:** `try: ... except ValueError: ...` — ловите именно `ValueError`, а не всё подряд. `None` проверьте до `try`: `float(None)` бросает `TypeError`, а не `ValueError`. `x.split()` делит строку по любым пробелам, включая неразрывный.

In [28]:
RAW = ["199.00", "1 499,50", None, "—", "349,90", "2\u00a0100", "", "  89.10  "]

# Во втором значении обычный пробел, в шестом — неразрывный (\u00a0).
# Если assert упал — RAW испортился при копировании: верните ячейку из стартового ноутбука.
assert RAW[1][1] == " " and RAW[5][1] == "\u00a0"


def to_value(x: str | None) -> float | None:
    # верните число или None; падать нельзя
    if x is None:
        return None

    text = x.strip()
    if text == "" or text == "—":
        return None

    text = "".join(text.split())

    text = text.replace(",", ".")

    try:
        return float(text)
    except ValueError:
        return None


def task7(raw: list) -> list:
    # верните [сумма разобранного, сколько отброшено]
    parsed = [to_value(x) for x in raw]
    good = [v for v in parsed if v is not None]
    dropped = len(parsed) - len(good)
    total = float(round(sum(good), 2)) if good else 0.0
    return [total, dropped]


answer_7 = task7(RAW)
answer_7

[4237.5, 3]

**Вопрос 7.** **Что опаснее.** Представьте ежедневный отчёт о выручке, построенный на этой функции. Что опаснее: функция, которая падает на `'—'`, или функция, которая молча возвращает `None`? Что бы вы добавили в отчёт, чтобы потеря была видна руководителю? Опирайтесь на свой ответ задачи 7: сколько из восьми значений было бы потеряно.

In [29]:
# Числа для ответа на вопрос 7 считайте здесь — по rows и answer_7.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
print("Всего значений в RAW:", len(RAW))
print("Разобрано:", len(RAW) - answer_7[1], ", отброшено:", answer_7[1])
print("Сумма разобранного:", answer_7[0])

Всего значений в RAW: 8
Разобрано: 5 , отброшено: 3
Сумма разобранного: 4237.5


#### ✍️ Ответ на вопрос 7

_Функция, которая молча возвращает None опаснее для отчёта, тк она просто тихо занижает показатели, но и падающая функция для ежедневного отчёта - не лучший вариант. Важно и не ломать отчёт, и сообщать о потерях, для этого необходимо как минимум добавить количество отброшеных значений, а в идеале ещё и причины их отбрасывания (None, прочерк и пустая строка). Сейчас отброшены 3 значения из 8 (37,5%), и отсутствие информации об этом могло бы повлиять, например, на восприятие среднего (4237.5/5 = 847.5 vs 4237.5/8 = 529.6875), тк без доп указаний читателю непонятно, какой знаменатель берётся в расчёте._

### Задача 8. Поправка на день недели

Аналитики из соседней команды прислали коэффициенты дней недели, чтобы сравнивать недели между собой. Прислали только будни — как это обычно и бывает.

**Что посчитать**

1. Для каждого дня недели возьмите сумму `value` по строкам с `ok = True` — как в задаче 3, но **до округления**.
2. Приведите её к `float` и умножьте на коэффициент дня из `WEIGHTS`. Если дня в справочнике нет — коэффициент `1.0`.
3. Сложите все произведения. Округлите до двух знаков **только итог**.

**Формат ответа:** одно число `float` до двух знаков, например `123456.78`.

**Подсказка:** `WEIGHTS.get(day, 1.0)` возвращает коэффициент или `1.0`, если дня нет. `Decimal * float` падает с `TypeError` — сначала `float(...)`.

In [30]:
WEIGHTS = {"Mon": 1.00, "Tue": 1.05, "Wed": 1.05, "Thu": 1.00, "Fri": 0.95}


def task8(rows: list[dict]) -> float:
    # верните взвешенную сумму
    sums = defaultdict(Decimal)
    for row in rows:
        if row["ok"]:
            sums[WEEKDAYS[row["day"].weekday()]] += row["value"]

    weighted_total = 0.0
    for day, total in sums.items():
        weighted_total += float(total) * WEIGHTS.get(day, 1.0)
    
    return round(weighted_total, 2)


answer_8 = task8(rows)
answer_8

36364.17

**Вопрос 8.** **Что изменила поправка.** Какие дни недели пошли с коэффициентом по умолчанию? На сколько процентов (с точностью до десятых) взвешенная сумма отличается от обычной суммы из задачи 1 — процент считайте от обычной суммы? В какой ситуации на данных ПРАЙМ такой `.get` с умолчанием спрятал бы ошибку, вместо того чтобы о ней сообщить?

In [31]:
# Числа для ответа на вопрос 8 считайте здесь — по rows и answer_8.
# Ячейка тоже выполняется на контрольном участке: даты, id и сегмент руками не вписывайте.
days_present = {WEEKDAYS[row["day"].weekday()] for row in rows if row["ok"]}
default_days = sorted(d for d in days_present if d not in WEIGHTS)

plain_sum = answer_1[2]
percent_diff = (answer_8 - plain_sum) / plain_sum * 100

print("Дни недели, встретившиеся в данных:", sorted(days_present))
print("Дни с коэффициентом по умолчанию (1.0, т.к. их нет в WEIGHTS):", default_days)
print("Обычная сумма (задача 1):", plain_sum, ", взвешенная (задача 8):", answer_8)
print("Отличие в процентах от обычной суммы:", round(percent_diff, 1))

Дни недели, встретившиеся в данных: ['Fri', 'Mon', 'Sat', 'Sun', 'Thu', 'Tue', 'Wed']
Дни с коэффициентом по умолчанию (1.0, т.к. их нет в WEIGHTS): ['Sat', 'Sun']
Обычная сумма (задача 1): 36010.43 , взвешенная (задача 8): 36364.17
Отличие в процентах от обычной суммы: 1.0


#### ✍️ Ответ на вопрос 8

_Суббота и воскресенье пошли с коэффициентами по умолчанию, тк их не было в списке весов. Взвешенная сумма отличается от обычной на 1% (2 дня с завышенным коэффициентом, 1 с заниженным, поэтому разница маленькая). .get мог спрятать ошибку в случае опечатки или пропуска значения. Найти ошибку можно было бы с помощью вывода списка дней, посчитанных по умолчанию._

## Часть 3. Итог — готовый код

При **Restart Kernel and Run All Cells** эта ячейка выполняется после всех задач: перед сдачей посмотрите её вывод. Она показывает ваши ответы и проверяет только формат, но не правильность.

Последняя строка вывода — служебная: по ней преподаватель сверяет ответы автоматически. Ячейку не удаляйте.

In [32]:
import json


def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


# Участок из ячейки 1.1 и ответы answer_1 … answer_8; у каждого ответа проверяем формат.
params = {"table_name": TABLE, "segment_column": SEGMENT_COLUMN, "segment": SEGMENT,
          "period_start": PERIOD_START, "period_end": PERIOD_END}
answers = {n: globals().get(f"answer_{n}") for n in range(1, 9)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n}: {answer!r}")
    print("   ✅ формат в порядке" if formats[n] is None else f"   ❌ {formats[n]}")

print("\nНе забудьте ответы словами — ячейки «✍️ Ответ на вопрос N» в части 2.")
print("\n--- служебная строка, не удаляйте ---")
print("HW1_ANSWERS=" + json.dumps(
    {"slice": params, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

задача 1: [1317, 1315, 36010.43, 30]
   ✅ формат в порядке
задача 2: 0
   ✅ формат в порядке
задача 3: {'Mon': 4664.1, 'Tue': 5598.15, 'Wed': 6900.77, 'Thu': 4357.58, 'Fri': 5424.23, 'Sat': 4518.13, 'Sun': 4547.47}
   ✅ формат в порядке
задача 4: ['3581c3e6-208a-44b8-b1e2-8e60caa613e4', 6]
   ✅ формат в порядке
задача 5: [['2025-04-11', 1724.82], ['2025-04-25', 1636.97], ['2025-04-16', 1635.98]]
   ✅ формат в порядке
задача 6: [16, '2025-04-16']
   ✅ формат в порядке
задача 7: [4237.5, 3]
   ✅ формат в порядке
задача 8: 36364.17
   ✅ формат в порядке

Не забудьте ответы словами — ячейки «✍️ Ответ на вопрос N» в части 2.

--- служебная строка, не удаляйте ---
HW1_ANSWERS={"slice": {"table_name": "service_usage", "segment_column": "device_type", "segment": "desktop", "period_start": "2025-04-01", "period_end": "2025-04-30"}, "answers": {"1": [1317, 1315, 36010.43, 30], "2": 0, "3": {"Mon": 4664.1, "Tue": 5598.15, "Wed": 6900.77, "Thu": 4357.58, "Fri": 5424.23, "Sat": 4518.13, "Sun": 4547

## Часть 4. Исследование — для тех, кто хочет 9 или 10

**Эта часть необязательна.** Части 1–3 — это оценка до 8 баллов, и 8 — это «отлично». 9 и 10 ставятся только за исследование: два задания, каждое до 1 балла сверху. Засчитываются, если основная часть набрала не меньше 15 баллов из 20.

Здесь нет готового формата ответа и заготовок. Вы сами решаете, что посчитать, считаете это кодом в ячейках ниже и пишете вывод. Ассистент может помочь с кодом, но ответ зависит от того, что вы увидите в своих данных. Текст вывода — ваш: правило про LLM действует и здесь.

| Каждое исследование оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке, а не вписаны руками | 0,4 |
| вывод следует из чисел, названо, чего ваш расчёт **не** доказывает | 0,3 |
| код исследования выполняется у преподавателя целиком — в том числе на контрольном участке | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Код исследования, как и всё остальное, считайте от значений ячейки 1.1: на контрольном участке он должен отработать без правок.

База умеет считать сама: `count(*)`, `count(distinct …)`, `sum(…)`, условие внутри агрегата — `count(*) filter (where …)`, разбивка — `group by`. Весь сегмент в Python не выгружайте: запрос дольше 2 минут сервер оборвёт. Функция `query` и словарь `COLUMNS` из части 1 здесь пригодятся.

### Исследование А. Можно ли верить вашей тысяче?

Руководитель прочитал сводку и спрашивает: «Вы смотрели тысячу участников, а в сегменте их гораздо больше. Насколько ваши числа верны для всего сегмента? Если в следующем месяце value на участника упадёт на 5 %, мы это заметим по такой тысяче?»

**Что сделать**

1. По **всему** вашему сегменту за окно — одним запросом к базе, не выгружая строки — посчитайте: сколько участников, долю строк с `ok = True`, `value` на участника (сумма `value` по строкам с `ok = True`, делённая на число различных участников).
2. Те же показатели — по вашей тысяче, из `rows`.
3. Возьмите **не меньше 20 других тысяч** участников того же сегмента и окна и посчитайте показатели для каждой.
4. Ответьте: насколько ваша тысяча отличается от всего сегмента? Каков разброс между тысячами? Заметно ли по одной тысяче падение `value` на участника на 5 % — и сколько участников нужно брать, чтобы такое падение было заметно?

**Подсказки.** Другая тысяча — это другое перемешивание. Ячейку 1.4 не меняйте — сделайте копию запроса в своей ячейке: `salted_sql = SLICE_SQL.replace("md5(actor_id)", "md5(actor_id || :salt)").format(table=TABLE, **COLUMNS[TABLE])` и вызывайте `query(salted_sql, segment=SEGMENT, period_start=PERIOD_START, period_end=PERIOD_END, salt=str(i))` с разными `i`. Разброс удобно описать стандартным отклонением (`statistics.pstdev`). Как разброс зависит от размера выборки — поищите «стандартная ошибка среднего».

In [33]:
# Исследование А: ваш код. Всё — от значений ячейки 1.1.
import math
import statistics

full_sql = """
select count(distinct {actor})                        as actors,
       count(*)                                        as n_rows,
       count(*) filter (where {ok})                    as ok_rows,
       coalesce(sum({value}) filter (where {ok}), 0)   as ok_value
from prime.{table}
where {segment_column} = :segment
  and {moment} >= cast(:period_start as date)
  and {moment} <  cast(:period_end as date) + 1
""".format(table=TABLE, **COLUMNS[TABLE])

full = query(full_sql, segment=SEGMENT, period_start=PERIOD_START, period_end=PERIOD_END)[0]
full_metrics = {
    "actors": full["actors"],
    "share_ok": full["ok_rows"] / full["n_rows"],
    "value_per_actor": float(full["ok_value"]) / full["actors"],
}

def slice_metrics(rs):
    actors = len({r["actor_id"] for r in rs})
    share_ok = sum(1 for r in rs if r["ok"]) / len(rs)
    value_per_actor = float(sum(r["value"] for r in rs if r["ok"])) / actors
    return {"actors": actors, "share_ok": share_ok, "value_per_actor": value_per_actor}

mine_metrics = slice_metrics(rows)

N_OTHER = 25
salted_sql = SLICE_SQL.replace("md5(actor_id)", "md5(actor_id || :salt)").format(
    table=TABLE, **COLUMNS[TABLE])

other_metrics = []
for i in range(1, N_OTHER + 1):
    rs = query(salted_sql, segment=SEGMENT, period_start=PERIOD_START,
               period_end=PERIOD_END, salt=str(i))
    other_metrics.append(slice_metrics(rs))
    print(f"тысяча {i}/{N_OTHER} готова", end="\r")

all_thousands = [mine_metrics] + other_metrics

print("\nВесь сегмент: участников", full_metrics["actors"])
print(f"{'Показатель':<18}{'Весь сегмент':>14}{'Моя тысяча':>14}{'Отличие, %':>12}"
      f"{'Среднее по тысячам':>15}{'sd по тысячам':>11}{'sd/среднее, %':>15}")

mean_v = sd_v = None
for name in ("share_ok", "value_per_actor"):
    vals = [m[name] for m in all_thousands]
    whole, mine = full_metrics[name], mine_metrics[name]
    mean_, sd_ = statistics.mean(vals), statistics.pstdev(vals)
    print(f"{name:<18}{whole:>14.4f}{mine:>14.4f}{(mine - whole) / whole * 100:>12.2f}"
          f"{mean_:>15.4f}{sd_:>11.4f}{sd_ / mean_ * 100:>15.2f}")
    if name == "value_per_actor":
        mean_v, sd_v = mean_, sd_

def norm_cdf(z):
    return 0.5 * (1 + math.erf(z / math.sqrt(2)))

def norm_ppf(p, lo=-8.0, hi=8.0, tol=1e-10):
    while hi - lo > tol:
        mid = (lo + hi) / 2
        if norm_cdf(mid) < p:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

Z_ALPHA = norm_ppf(0.975)
Z_BETA_80 = norm_ppf(0.80)

n_current = mine_metrics["actors"]

sigma_hat = sd_v * math.sqrt(n_current)

def se_at(n):
    return sigma_hat / math.sqrt(n)

def power_one_sample(drop_share, n):
    """Вероятность заметить падение value_per_actor на drop_share при выборке n участников."""
    drop = drop_share * mean_v
    z = drop / se_at(n) - Z_ALPHA
    return norm_cdf(z)

print("\nС какой вероятностью падение будет определено:")
for n in (n_current, 2000, 5000, full_metrics["actors"]):
    row = f"  n={n}:"
    for drop_share in (0.01, 0.03, 0.05, 0.10):
        row += f"  {drop_share*100:>3.0f}%->{power_one_sample(drop_share, n)*100:5.1f}%"
    print(row)

def n_needed_one_sample(drop_share, z_a=Z_ALPHA, z_b=Z_BETA_80):
    """Сколько участников нужно, чтобы с вероятностью 80% отличить падение drop_share от уже известного уровня (весь сегмент)."""
    drop = drop_share * mean_v
    return math.ceil(((z_a + z_b) * sigma_hat / drop) ** 2)

def n_needed_two_sample(drop_share, z_a=Z_ALPHA, z_b=Z_BETA_80):
    """То же самое, но сравниваем два периода между собой, без опоры на прошлый уровень как на точную константу, отсюда множитель 2 под корнем."""
    drop = drop_share * mean_v
    return math.ceil(2 * ((z_a + z_b) * sigma_hat / drop) ** 2)

print("\nТочный расчёт нужного n (α=0.05, вероятность 80%):")
for drop_share in (0.01, 0.03, 0.05, 0.10):
    n1 = n_needed_one_sample(drop_share)
    n2 = n_needed_two_sample(drop_share)
    print(f"  падение {drop_share*100:>4.0f}%: против известного уровня ~{n1}, "
          f"два периода без опоры на прошлый уровень ~{n2} на каждый")

def median_value_per_actor(rs):
    by_actor = {}
    for r in rs:
        if r["ok"]:
            by_actor[r["actor_id"]] = by_actor.get(r["actor_id"], 0.0) + float(r["value"])
    return statistics.median(by_actor.values()) if by_actor else 0.0

med_mine = median_value_per_actor(rows)
mean_mine = mine_metrics["value_per_actor"]
print(f"\nМоя тысяча: среднее value/участника {mean_mine:.3f}, медиана {med_mine:.3f}, "
      f"среднее/медиана = {mean_mine/med_mine:.3f}")

тысяча 25/25 готова
Весь сегмент: участников 45504
Показатель          Весь сегмент    Моя тысяча  Отличие, %Среднее по тысячамsd по тысячам  sd/среднее, %
share_ok                  0.9981        0.9985        0.04         0.9983     0.0012           0.12
value_per_actor          34.9643       36.0104        2.99        35.1649     0.9616           2.73

С какой вероятностью падение будет определено:
  n=1000:    1%->  5.5%    3%-> 19.4%    5%-> 44.8%   10%-> 95.5%
  n=2000:    1%->  7.5%    3%-> 34.1%    5%-> 73.4%   10%-> 99.9%
  n=5000:    1%-> 12.7%    3%-> 68.9%    5%-> 98.3%   10%->100.0%
  n=45504:    1%-> 69.4%    3%->100.0%    5%->100.0%   10%->100.0%

Точный расчёт нужного n (α=0.05, вероятность 80%):
  падение    1%: против известного уровня ~58693, два периода без опоры на прошлый уровень ~117386 на каждый
  падение    3%: против известного уровня ~6522, два периода без опоры на прошлый уровень ~13043 на каждый
  падение    5%: против известного уровня ~2348, два периода бе

#### ✍️ Исследование А

_Чтобы понять, насколько моей тысяче участников можно доверять как оценке всего сегмента, я сравнила её метрики с показателями всего сегмента (45504 участника) и с 25 другими случайными тысячами, а также посчитала, с какой вероятностью выборка размером 1000 позволила бы определить реальное падение средней длительности на участника. По доле состоявшихся сессий моя тысяча практически не отличается от остальных (разница 0,04%), а по средней длительности на участника отличие составляет 2,99%, что сопоставимо с обычным разбросом между случайными тысячами (2,73%). При этом выборка из тысячи участников позволит определить падение на 5% только в 44,8% случаев. Чтобы заметить его с большей вероятностью (80%), нужно минимум 2348 участников (5,2% от всего сегмента). Среднее (36,01) в 1,554 раза больше медианы (23,18), это сильный перекос, а это значит, что у части участников сессии значительно длиннее остальных, из-за чего типичный разброс возрастает и требует большего n. Поэтому может быть полезно мониторить медиану вместо среднего: она устойчивее к выбросам и, скорее всего, даст лучшую вероятность при том же n._

_Расчёт основан на 25 тысячах из одного и того же месяца - апреля, то есть меняются только сами участники, а не время. А в реальности сравнивать обычно приходится два разных периода, например, апрель с маем, где есть ещё и обычная изменчивость по дням (по дням недели и конкретным датам, как было видно в заданиях раньше). Эту изменчивость расчёт не учитывал, поэтому на практике разброс между реальными периодами может быть больше 2,73%, а нужное n больше 2348._

### Исследование Б. Ваш сегмент особенный?

Маркетинг заметил, что в одних сегментах участники приносят заметно больше, чем в других, и хочет перераспределить бюджет в пользу «ценных» сегментов. Руководитель просит проверить: правда ли ваш сегмент отличается от соседних — и чем именно.

**Что сделать**

1. Для **всех сегментов** вашей таблицы за ваше окно посчитайте: число участников, строк на участника, долю строк с `ok = True`, `value` на одну строку с `ok = True` и `value` на участника.
2. Разложите `value` на участника на множители и найдите, какой из них даёт различие между сегментами.
3. Проверьте, держится ли картина в предыдущем окне той же длины. Данные начинаются 6 января 2025 года: если предыдущее окно начинается раньше, оно неполное — тогда сравнивайте доли и значения на участника или на строку, а не число участников и строк.
4. Вывод для маркетинга: «ценный» ли ваш сегмент; что на самом деле различается; какое решение по бюджету из этого следует — и чего ваши данные не доказывают.

**Подсказка.** Здесь нужны не выгрузка `rows`, а агрегаты по всей таблице за окно с разбивкой `group by` колонке сегмента.

In [34]:
# Исследование Б: ваш код. Всё — от значений ячейки 1.1.
import math
from datetime import timedelta

DATA_START = date(2025, 1, 6)

seg_sql = """
select {segment_column}                                   as segment,
       count(distinct {actor})                            as actors,
       count(*)                                           as n_rows,
       count(*) filter (where {ok})                       as ok_rows,
       coalesce(sum({value}) filter (where {ok}), 0)      as ok_value
from prime.{table}
where {moment} >= cast(:period_start as date)
  and {moment} <  cast(:period_end as date) + 1
group by {segment_column}
""".format(table=TABLE, **COLUMNS[TABLE])

def segment_table(start, end):
    """Показатели по каждому сегменту таблицы за окно [start, end] включительно."""
    table = []
    for r in query(seg_sql, period_start=start, period_end=end):
        actors, n_rows, ok_rows = r["actors"] or 1, r["n_rows"], r["ok_rows"]
        ok_value = float(r["ok_value"])
        table.append({
            "segment": r["segment"], "actors": actors, "n_rows": n_rows,
            "rows_per_actor": n_rows / actors,
            "share_ok": ok_rows / n_rows,
            "value_per_ok_row": ok_value / ok_rows if ok_rows else 0.0,
            "value_per_actor": ok_value / actors,
        })
    return sorted(table, key=lambda t: -t["value_per_actor"])

def show(table, title):
    print(f"\n{title}")
    print(f"{'Сегмент':<14}{'Участн.':>9}{'Строк/уч.':>11}{'Доля ok':>9}"
          f"{'value/строку':>14}{'value/уч.':>12}")
    for t in table:
        mark = "  <- мой" if t["segment"] == SEGMENT else ""
        print(f"{str(t['segment']):<14}{t['actors']:>9}{t['rows_per_actor']:>11.3f}"
              f"{t['share_ok']:>9.3f}{t['value_per_ok_row']:>14.3f}{t['value_per_actor']:>12.3f}{mark}")

def decompose(table, title):
    """value/участника = строк/участника x доля ok x value/строку.
    Сравниваем каждый сегмент с моим: три отношения перемножаются в отношение value/участника."""
    base = next(t for t in table if t["segment"] == SEGMENT)
    print(f"\n{title}: отношение к моему сегменту ({SEGMENT})")
    print(f"{'Сегмент':<14}{'Строк/уч.':>11}{'Доля ok':>9}{'value/строку':>14}{'Итого':>9}  Наибольшее отличие")
    for t in table:
        if t is base:
            continue
        factors = {
            "Строк на участника": t["rows_per_actor"] / base["rows_per_actor"],
            "Доля ok": t["share_ok"] / base["share_ok"],
            "value на строку": (t["value_per_ok_row"] / base["value_per_ok_row"]
                                if base["value_per_ok_row"] else float("nan")),
        }
        total = t["value_per_actor"] / base["value_per_actor"] if base["value_per_actor"] else float("nan")
        valid = {k: v for k, v in factors.items() if v > 0}
        main = max(valid, key=lambda k: abs(math.log(valid[k]))) if valid else "—"
        print(f"{str(t['segment']):<14}{factors['Строк на участника']:>11.3f}{factors['Доля ok']:>9.3f}"
              f"{factors['value на строку']:>14.3f}{total:>9.3f}  {main}")

now_table = segment_table(PERIOD_START, PERIOD_END)
show(now_table, f"Все сегменты таблицы {TABLE}, окно {PERIOD_START} — {PERIOD_END}")
decompose(now_table, "Текущее окно")

start_d, end_d = date.fromisoformat(PERIOD_START), date.fromisoformat(PERIOD_END)
days = (end_d - start_d).days + 1
prev_end = start_d - timedelta(days=1)
prev_start = prev_end - timedelta(days=days - 1)
if prev_start < DATA_START:
    print(f"\nПредыдущее окно {prev_start} — {prev_end} начинается раньше данных ({DATA_START}) и неполное")

prev_table = segment_table(prev_start.isoformat(), prev_end.isoformat())
show(prev_table, f"Предыдущее окно {prev_start} — {prev_end}")
decompose(prev_table, "Предыдущее окно")

def rank_of_mine(table):
    return [t["segment"] for t in table].index(SEGMENT) + 1

print(f"\nМесто моего сегмента по value на участника: в текущем окне {rank_of_mine(now_table)} "
      f"из {len(now_table)}, в предыдущем - {rank_of_mine(prev_table)} из {len(prev_table)}")

skew_sql = """
with per_actor as (
    select {segment_column}                                   as segment,
           {actor}                                             as actor_id,
           sum(case when {ok} then {value} else 0 end)        as actor_value
    from prime.{table}
    where {moment} >= cast(:period_start as date)
      and {moment} <  cast(:period_end as date) + 1
    group by {segment_column}, {actor}
)
select segment,
       count(*)                                        as actors,
       avg(actor_value)                                 as mean_value,
       percentile_cont(0.5) within group (order by actor_value) as median_value
from per_actor
group by segment
""".format(table=TABLE, **COLUMNS[TABLE])

skew_rows = query(skew_sql, period_start=PERIOD_START, period_end=PERIOD_END)
skew_table = sorted(
    ({"segment": r["segment"], "actors": r["actors"],
      "mean": float(r["mean_value"]), "median": float(r["median_value"]),
      "ratio": float(r["mean_value"]) / float(r["median_value"]) if r["median_value"] else float("nan")}
     for r in skew_rows),
    key=lambda t: -t["ratio"]
)

print("\nПерекос (среднее/медиана value на участника) по сегментам:")
print(f"{'Сегмент':<14}{'Участн.':>9}{'Среднее':>10}{'Медиана':>10}{'Среднее/медиана':>17}")
for t in skew_table:
    mark = "  <- мой" if t["segment"] == SEGMENT else ""
    print(f"{str(t['segment']):<14}{t['actors']:>9}{t['mean']:>10.2f}{t['median']:>10.2f}"
          f"{t['ratio']:>17.3f}{mark}")

null_sql = """
select count(*)                    as null_rows,
       count(distinct {actor})     as null_actors
from prime.{table}
where {segment_column} is null
  and {moment} >= cast(:period_start as date)
  and {moment} <  cast(:period_end as date) + 1
""".format(table=TABLE, **COLUMNS[TABLE])

null_info = query(null_sql, period_start=PERIOD_START, period_end=PERIOD_END)[0]
print(f"\nСтрок без сегмента (segment_column IS NULL) за окно: {null_info['null_rows']}, "
      f"участников: {null_info['null_actors']}")
if null_info["null_rows"]:
    print("Эти строки не попадают ни в одну строку таблицы по сегментам выше, картина по сегментам не покрывает их вообще.")
else:
    print("Пропусков в сегменте нет, таблица по сегментам покрывает все строки окна.")

my_row = next(t for t in skew_table if t["segment"] == SEGMENT)
print(f"\nМой сегмент ({SEGMENT}): {my_row['actors']} участников, "
      f"среднее value на участника {my_row['mean']:.2f}")
print("Если бы мой сегмент показывал value на участника как другой, суммарная разница составила бы:")
for t in skew_table:
    if t is my_row:
        continue
    diff_per_actor = t["mean"] - my_row["mean"]
    total_diff = diff_per_actor * my_row["actors"]
    print(f"  как у {t['segment']}: {diff_per_actor:+.2f} на участника x {my_row['actors']} "
          f"участников = {total_diff:+.0f}")


Все сегменты таблицы service_usage, окно 2025-04-01 — 2025-04-30
Сегмент         Участн.  Строк/уч.  Доля ok  value/строку   value/уч.
mobile           101743      1.866    0.998        27.040      50.361
desktop           45504      1.291    0.998        27.139      34.964  <- мой
tv                33520      1.203    0.998        27.051      32.465
tablet            19537      1.116    0.998        27.281      30.382

Текущее окно: отношение к моему сегменту (desktop)
Сегмент         Строк/уч.  Доля ok  value/строку    Итого  Наибольшее отличие
mobile              1.446    1.000         0.996    1.440  Строк на участника
tv                  0.932    1.000         0.997    0.929  Строк на участника
tablet              0.865    1.000         1.005    0.869  Строк на участника

Предыдущее окно 2025-03-02 — 2025-03-31
Сегмент         Участн.  Строк/уч.  Доля ok  value/строку   value/уч.
mobile            96345      1.859    0.998        26.975      50.039
desktop           42983      1.

#### ✍️ Исследование Б

_Чтобы понять, правда ли мой сегмент десктопа отличается от других сегментов и чем именно, я сделала проверку по всем сегментам таблицы за текущее и предыдущее окно: посчитала число участников, строк на участника, долю состоявшихся сессий, длительность на строку и на участника, а также перекос среднее/медиана. Получилось: десктоп на 2 месте из 4 по длительности на участника, причём в обоих окнах (34.96 в апреле и 35.08 в марте), то есть картина стабильная. Доля состоявшихся сессий и длительность сессии у всех сегментов почти одинаковые (0.997-0.998 и 27.04-27.28 мин), а число сессий на участника у мобайла заметно выше (1.866 против 1.291 у десктопа), из-за этого и вся разница в длительности на участника. Перекос у десктопа не выше, чем у остальных, значит разница не из-за нескольких активных участников, а типичная для сегмента в целом. Если бы десктоп давал длительность на участника как мобайл, разница составила бы около 700629 на 45504 участниках, то есть и в абсолютных числах это немало. Значит, десктоп уступает не по качеству или длительности сессий, а по тому, как часто заходят, поэтому бюджет логичнее направлять на увеличение частоты использования, а не на удержание в рамках уже начатой сессии. При этом расчёт не доказывает причинно-следственную связь: неизвестно, дело в самом устройстве или в том, что на десктоп изначально заходит менее вовлечённая аудитория._

## Напоследок

Что вас удивило в вашем участке? Одна-две фразы в ячейке ниже — по желанию и без баллов.

И пройдите, пожалуйста, **[короткий опрос о том, как вам это задание](https://forms.gle/5tNT4P4diHVyzhdFA)** — пара минут. Следующие задания соберу с учётом ответов.

#### ✍️ Что удивило

_Всё-таки удивили пики в пятницы_